# Nehalennia data layer

Harvests each source into a dataframe, then writes the columns RMO and RMA share to
`presentation_layer/data/objecten.json`.

In [ ]:
# %pip install pandas openpyxl rdflib

from pathlib import Path

import pandas as pd

import collectie_zeeland
import perioden
import rma
import rmo

OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)

OBJECTEN_PATH = Path("../presentation_layer/data/objecten.json")

# carried even though only RMO has them, so RMA rows get them as null
EXTRA_COLUMNS = ["gevonden_datum_van", "gevonden_datum_tot"]


def write_json(path, frames, extra=EXTRA_COLUMNS):
    """The columns every frame shares plus `extra`, stacked into one JSON array."""
    shared = [column for column in frames[0].columns
              if all(column in frame.columns for frame in frames)]
    columns = shared + [column for column in extra if column not in shared]
    objecten = pd.concat([frame.reindex(columns=columns) for frame in frames],
                         ignore_index=True)

    # reindex fills absent columns with float NaN, which would turn 1970 into 1970.0
    for column in columns:
        owners = [frame[column] for frame in frames if column in frame.columns]
        if any(pd.api.types.is_integer_dtype(owner) for owner in owners):
            objecten[column] = objecten[column].astype("Int64")

    path.parent.mkdir(parents=True, exist_ok=True)
    objecten.to_json(path, orient="records", indent=2, force_ascii=False)

    print(f"wrote {len(objecten)} objects x {len(columns)} columns to {path.resolve()}")
    return objecten

## RMO

First run reads ~400 object pages at one request every half second, so allow a few
minutes; after that everything comes from `out/raw/` and `out/pages/`.

`rmo.fetch` then matches the transcriptions against the hand-made
`source/edcs_nehalennia_analyse.xlsx`, adding `edcs_id`, `inschrijving`, `edcs_score`
and `edcs_vertaling`. The first three hold ` | `-joined candidate lists in matching
order, best first; `edcs_vertaling` is the Dutch draft translation of that best match.

In [ ]:
df_rmo, df_rmo_images = rmo.fetch(cache_dir=OUT_DIR)
df_rmo.head()

## Rijksmuseum (RMA)

Reads the hand-picked `source/rma_nehalennia.csv`, follows every pid into its Linked Art
record for the IIIF image, and joins the EDCS links already in `inscripties` to their
Dutch draft translation. First run fetches 52 records at one request every half second.

In [ ]:
df_rma = rma.fetch(cache_dir=OUT_DIR)
df_rma[["objectnummer", "titel", "image_url", "edcs_vertaling"]].head()

## Collectie Zeeland

Streams the dump on every run (no cache), which takes about 5 seconds. Kept for the
Turtle export only; these objects are not part of `objecten.json`.

In [ ]:
df_zeeland = collectie_zeeland.fetch(ttl_path=OUT_DIR / "zeeland_nehalennia.ttl")
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger"]]

## Export

`perioden.assign` first places every object on the timeline: a period matches when one
of the object's date spans overlaps its `datum_van`..`datum_tot`. RMO objects have two
spans, their dating and their find date, so they can land in more than one period.

The columns RMO and RMA have in common, plus `EXTRA_COLUMNS`, then travel to the
presentation layer. RMA has no find date, so those two are null on its rows.

In [ ]:
df_rmo = perioden.assign(df_rmo)
df_rma = perioden.assign(df_rma)

objecten = write_json(OBJECTEN_PATH, [df_rmo, df_rma])
objecten.head()